In [1]:
!nvidia-smi

Sat May 30 06:47:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Install zstd for faster decompression of datasets
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (8,570 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [26]:
import subprocess
import time
import urllib.request

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait until the server is ready (up to 15 seconds)
for _ in range(15):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("Ollama server is ready")
        break
    except Exception:
        time.sleep(1)
else:
    print("ERROR: Ollama server did not start. Try re-running this cell.")

Ollama server is ready


In [6]:
!ollama pull qwen3.5:4b

In [7]:
%pip install -q ollama

In [8]:
import ollama

response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "What is the capital of Japan?"}]
)

print(response["message"]["content"])

The capital of Japan is **Tokyo**.


In [9]:
%pip install -q openai

In [10]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

response = client.chat.completions.create(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "日本の首都はどこですか？"}]
)

print(response.choices[0].message.content)

日本の首都は**東京**です。


In [11]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

response = client.chat.completions.create(
    model="qwen3.5:4b",
    messages=[
        {"role": "system", "content": "あなたは親切な日本語アシスタントです。必ず日本語で答えてください。"},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print(response.choices[0].message.content)

はい、ご質問ありがとうございます。

フランスの首都は、パリ（Paris）です。

その他、気になることがあればいつでもお気軽にお尋ねくださいね！


In [12]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

messages = [
    {"role": "system", "content": "あなたは親切な日本語アシスタントです。必ず日本語で答えてください。"},
]

turns = [
    "私の名前は太郎です。",
    "私の名前は何ですか？",
    "私の名前をローマ字で書いてください。",
]

for user_input in turns:
    messages.append({"role": "user", "content": user_input})
    print(f"User: {user_input}")

    response = client.chat.completions.create(
        model="qwen3.5:4b",
        messages=messages,
    )

    assistant_reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_reply})
    print(f"Assistant: {assistant_reply}\n")

User: 私の名前は太郎です。
Assistant: こんにちは、太郎さん！
お名前をおっしゃっていただき、ありがとうございます。
何かお手伝いできることがありますでしょうか？
何かご用件やお尋ねのことがございましたら、お気軽にお申し付けください。
よろしくおねがいします！

User: 私の名前は何ですか？
Assistant: はい、私のメモによると、お名前はいわゆる「太郎」さんですね。
少し変なことを言ったようですが、ご質問の回答として、私はあなたのお名前「太郎」ということです。何かお手伝いできますか？

User: 私の名前をローマ字で書いてください。
Assistant: はい、承知いたしました。

お名前「太郎」のローマ字は、通常は「Taro」と表記するのが一般的ですね。
何か他にもご用件やお尋ねのことがございましたら、お気軽にお申し付けください。

よろしくお願いいたします！



In [13]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

stream = client.chat.completions.create(
    model="qwen3.5:4b",
    messages=[
        {"role": "system", "content": "あなたは親切な日本語アシスタントです。必ず日本語で答えてください。"},
        {"role": "user", "content": "日本の四季について説明してください。"}
    ],
    stream=True,
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

こんにちは！日本の四季（しんき・しんき）は、非常に美しい特徴があり、日本人にとってとても大切なものです。

主な特徴について、簡単に説明させていただきますね。

**1. 春（はる）**
春は、一年がスタートする時期で最も心地よい季節です。新緑（しんりょく）が始まり、桜（さくら）が満開になるのが代表格です。多くの人は、桜の花を見るために「花見（はなみ）」を楽しむ文化があります。

**2. 夏（なつ）**
夏は、太陽の光が強くなり暑さが強くなる季節です。雨が多い時期は、「梅雨（つゆ）」と呼ばれ、湿度が高く少し不気味な雰囲気があります。台風が来ると、激しい雷雨や強風が楽しめます。

**3. 秋（あき）**
秋は、一年の最も美しい時期の一つとされています。空気が澄み合い、草木の葉が赤や金色に変色して「紅葉（こうよう）」が見られます。秋の朝は涼しくて、多くの人が散歩や紅葉狩りに出ます。

**4. 冬（とう）**
冬は、寒気を感じる季節で「雪」の時期が訪れます。北海道や山岳地帯では積雪が続き、スキーやスノボなどウィンタースポーツが楽しめます。また、年末や正月に伝統的な行事や祭りが行われます。

四季は私たちの生活や文化、祭り、食べ物も大きく影響しています。ぜひ、日本の四季の美しさを体感してみてください。

他に特に気になる季節のことについても、お手伝いできてしまいますよ！

In [14]:
import json
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

response = client.chat.completions.create(
    model="qwen3.5:4b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that responds only in valid JSON."},
        {"role": "user", "content": (
            "日本の主要都市を3つ挙げて、以下のJSON形式で返してください:\n"
            '{"cities": [{"name": "...", "prefecture": "...", "population": ...}]}'
        )}
    ],
    response_format={"type": "json_object"},
)

result = json.loads(response.choices[0].message.content)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "cities": [
    {
      "name": "東京",
      "prefecture": "東京都",
      "population": 13900000
    },
    {
      "name": "大阪",
      "prefecture": "大阪府",
      "population": 2800000
    },
    {
      "name": "京都",
      "prefecture": "京都府",
      "population": 1500000
    }
  ]
}


In [16]:
import json
import requests
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Define the tool
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "指定した都市の現在の天気を取得する",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name in English (e.g. Tokyo, Osaka)"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "温度の単位"},
                },
                "required": ["city"],
            },
        },
    }
]

def get_weather(city, unit="celsius"):
    # Step 1: Geocode city name to coordinates
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1}
    ).json()

    if not geo.get("results"):
        return json.dumps({"error": f"City '{city}' not found."}, ensure_ascii=False)

    location = geo["results"][0]
    lat, lon = location["latitude"], location["longitude"]

    # Step 2: Fetch current weather
    temperature_unit = "fahrenheit" if unit == "fahrenheit" else "celsius"
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,weathercode,windspeed_10m",
            "temperature_unit": temperature_unit,
        }
    ).json()

    current = weather["current"]
    return json.dumps({
        "city": city,
        "temperature": current["temperature_2m"],
        "unit": unit,
        "windspeed_kmh": current["windspeed_10m"],
        "weathercode": current["weathercode"],
    }, ensure_ascii=False)

# Step 1: Send user message with tools
messages = [{"role": "user", "content": "東京の天気を教えてください。"}]
response = client.chat.completions.create(model="qwen3.5:4b", messages=messages, tools=tools)

assistant_message = response.choices[0].message
messages.append(assistant_message)

# Step 2: Execute the tool call if requested
if assistant_message.tool_calls:
    for tool_call in assistant_message.tool_calls:
        args = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_call.function.name}({args})")
        result = get_weather(**args)
        print(f"Tool result: {result}\n")
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    # Step 3: Send tool result back to get final response
    final_response = client.chat.completions.create(model="qwen3.5:4b", messages=messages)
    print(f"Assistant: {final_response.choices[0].message.content}")
else:
    print(f"Assistant: {assistant_message.content}")

Tool called: get_weather({'city': 'Tokyo', 'unit': 'celsius'})
Tool result: {"city": "Tokyo", "temperature": 25.0, "unit": "celsius", "windspeed_kmh": 9.5, "weathercode": 0}

Assistant: # 東京の天気情報

**現在、東京の天気が以下の通りです。**

| 項目 | 値 |
|------|-----|
| **気温** | 25.0°C |
| **風速** | 9.5 km/h |
| **天気状態** | 快晴（晴れ） |

**備考：**
- **天気コード 0** は、一般的に「快晴」や「晴れ」という状態を表しています。
- 25度の気温は、東京の温暖な春先〜夏初日のような気温です。
- 風は少しありますが、快適な吹風があります。

今日、東京は晴れの良い日ですね。外出や户外活动は十分快適に感じられるかもしれません！🫡

もし、他にも知りたい情報（気温の更新など）が必要なら、お手伝いさせていただきます。


In [29]:
import re
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

def extract_answer(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

messages = [
    {"role": "system", "content": "You are a haiku poet. Reply with exactly one haiku (3 lines only). No explanations, no titles, no romanization."},
    {"role": "user",   "content": "俳句を一句詠んでください。"},
]

configs = [
    {"label": "temperature=0.0 (決定的・一貫性重視)",  "temperature": 0.0},
    {"label": "temperature=1.0 (バランス・デフォルト)", "temperature": 1.0},
    # {"label": "temperature=2.0 (ランダム・創造的)",    "temperature": 2.0},
    # {"label": "top_p=0.1 (保守的な候補に絞る)",        "temperature": 1.0, "top_p": 0.1},
    # {"label": "frequency_penalty=2.0 (繰り返し抑制)",  "temperature": 1.0, "frequency_penalty": 2.0},
]

for config in configs:
    label = config.pop("label")
    response = client.chat.completions.create(
        model="qwen3.5:4b",
        messages=messages,
        max_tokens=4096,
        **config,
    )
    print(f"[{label}]")
    print(extract_answer(response.choices[0].message.content))
    print()

[temperature=0.0 (決定的・一貫性重視)]


[temperature=1.0 (バランス・デフォルト)]




In [30]:
import json
import re
import requests
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "qwen3.5:4b"

def extract_answer(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

# ── 1. Router Agent ────────────────────────────────────────────────────────────
def router_agent(user_input: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=512,
        messages=[
            {"role": "system", "content": (
                'You are a routing agent. Classify the user input into exactly one label. '
                'Reply ONLY with valid JSON: {"label": "<label>"} '
                'Labels: "weather" (weather questions), "haiku" (creative/poem requests), "general" (everything else).'
            )},
            {"role": "user", "content": user_input},
        ],
        response_format={"type": "json_object"},
    )
    result = json.loads(extract_answer(response.choices[0].message.content))
    print(f"[Router] → label: {result['label']}")
    return result["label"]

# ── 2. Specialist Agents ───────────────────────────────────────────────────────
def weather_specialist(user_input: str) -> str:
    tools = [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name in English"},
                },
                "required": ["city"],
            },
        },
    }]

    def get_weather(city):
        geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                           params={"name": city, "count": 1}).json()
        if not geo.get("results"):
            return json.dumps({"error": f"City '{city}' not found."})
        loc = geo["results"][0]
        weather = requests.get("https://api.open-meteo.com/v1/forecast", params={
            "latitude": loc["latitude"], "longitude": loc["longitude"],
            "current": "temperature_2m,weathercode,windspeed_10m",
        }).json()
        c = weather["current"]
        return json.dumps({"city": city, "temperature_celsius": c["temperature_2m"],
                           "windspeed_kmh": c["windspeed_10m"]})

    messages = [{"role": "user", "content": user_input}]
    res = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, max_tokens=1024)
    msg = res.choices[0].message
    messages.append(msg)

    if msg.tool_calls:
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            tool_result = get_weather(**args)
            print(f"[Weather Specialist] tool call: get_weather({args})")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": tool_result})
        final = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=1024)
        return extract_answer(final.choices[0].message.content)
    return extract_answer(msg.content)

def haiku_specialist(user_input: str) -> str:
    res = client.chat.completions.create(
        model=MODEL, max_tokens=4096,
        messages=[
            {"role": "system", "content": "You are a haiku poet. Reply with exactly one haiku (3 lines only). No explanations."},
            {"role": "user", "content": user_input},
        ],
    )
    return extract_answer(res.choices[0].message.content)

def general_specialist(user_input: str) -> str:
    res = client.chat.completions.create(
        model=MODEL, max_tokens=2048,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer concisely in Japanese."},
            {"role": "user", "content": user_input},
        ],
    )
    return extract_answer(res.choices[0].message.content)

SPECIALISTS = {
    "weather": weather_specialist,
    "haiku":   haiku_specialist,
    "general": general_specialist,
}

# ── 3. Guardrail Agent ─────────────────────────────────────────────────────────
def guardrail_agent(specialist_output: str) -> str:
    res = client.chat.completions.create(
        model=MODEL, max_tokens=2048,
        messages=[
            {"role": "system", "content": (
                "You are a politeness guardrail. Rewrite the given response to be warm, "
                "polite, and natural in Japanese. Keep the content accurate. Output the rewritten response only."
            )},
            {"role": "user", "content": specialist_output},
        ],
    )
    return extract_answer(res.choices[0].message.content)

# ── Run the pipeline ───────────────────────────────────────────────────────────
def run_agent(user_input: str):
    print(f"User: {user_input}\n")
    label    = router_agent(user_input)
    raw      = SPECIALISTS.get(label, general_specialist)(user_input)
    print(f"[Specialist ({label})]\n{raw}\n")
    polished = guardrail_agent(raw)
    print(f"[Guardrail] Final response:\n{polished}")

# Test with different inputs
for query in ["東京の天気を教えてください。", "秋について俳句を詠んでください。", "日本で一番高い山は何ですか？"]:
    print("=" * 60)
    run_agent(query)
    print()


User: 東京の天気を教えてください。

[Router] → label: weather
[Weather Specialist] tool call: get_weather({'city': 'Tokyo'})
[Specialist (weather)]
現在、東京の天気は以下の通りです。

🌡️ 気温：23.9°C  
💨 風速：8.7 km/h  

今のところ、東京は比較的温暖的な天気で、風も穏やかなようです。外出される際は、軽い服装で快適に過ごせるでしょう。

[Guardrail] Final response:


User: 秋について俳句を詠んでください。

[Router] → label: haiku
[Specialist (haiku)]


[Guardrail] Final response:


User: 日本で一番高い山は何ですか？

[Router] → label: general
[Specialist (general)]
日本で一番高い山は富士山です。

[Guardrail] Final response:


